In [6]:
import pandas as pd
import numpy as np
import glob
import os

In [ ]:
# Heat detection sample logic on Cow data from 08th Feb to 13th Feb. The farm manager has confirmed heat detected on 13th.

data_path = '124'
#files = sorted(glob.glob(os.path.join(data_path, 'node-184-2026-02-*.csv')))
file_pattern = os.path.join(data_path, 'node-124-2026-01-*.csv')


import pandas as pd
import numpy as np
import glob
import os

class CattleIntelligenceSystem:
    def __init__(self):
        # Configuration for scalability and precision
        self.RES_THRESHOLD = 0.35
        self.NIGHT_HOURS = [23, 0, 1, 2, 3]
        self.SOLAR_HOURS = [11, 12, 13, 14, 15, 16]

    # --- BLOCK 1: FEATURE EXTRACTION ---
    def extract_features(self, df):
        df = df.copy()
        df['mag'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)
        df['vedba'] = np.abs(df['mag'] - df['mag'].rolling(window=50, center=True).mean())
        return df.fillna(0)

    # --- BLOCK 2: AR MODEL (Activity Recognition) ---
    def predict_activity(self, df):
        conditions = [(df['vedba'] > self.RES_THRESHOLD), (df['vedba'] > 0.15)]
        df['activity_class'] = np.select(conditions, ['RES', 'FEED'], default='STANDING')
        return df

    # --- BLOCK 3: ENVIRONMENTAL HEAT STRESS BLOCK ---
    def check_heat_stress(self, hourly_df):
        solar_data = hourly_df[hourly_df.index.hour.isin(self.SOLAR_HOURS)]
        # Flag environmental stress if daytime avg is extreme
        return solar_data['temp_mean'].mean() > 40.5

    # --- BLOCK 4: CATTLE LOGIC (The Core Detector) ---
    def cattle_logic_engine(self, hourly_df, env_stress):
        # GLOBAL ANCHOR: The lowest resting temperature of the whole period
        # This preserves the 9.11C delta reported for Feb 13
        global_night_base = hourly_df[hourly_df.index.hour.isin([0, 1, 2])]['temp_mean'].min()

        daily_stats = []
        for day, day_data in hourly_df.groupby(hourly_df.index.date):
            # 1. Biological Spike: Night Peak vs Global Anchor
            night_max = day_data[day_data.index.hour.isin(self.NIGHT_HOURS)]['temp_mean'].max()
            night_spike = night_max - global_night_base

            # 2. Behavioral Persistence: 3-hour sustained restlessness
            persistence = day_data['res_ratio'].rolling(3, min_periods=1).mean().max()

            # 3. Decision Score
            score = (night_spike * 15) + (persistence * 40)

            daily_stats.append({
                'date': str(day),
                'spike': max(0, night_spike),
                'res': persistence if not np.isnan(persistence) else 0,
                'score': score,
                'stress': env_stress
            })

        # Rank by score to identify the single most clinical day
        daily_stats.sort(key=lambda x: x['score'], reverse=True)
        peak_date = daily_stats[0]['date']
        return daily_stats, peak_date

    # --- BLOCK 5: ALERT & LOGGING ---
    def generate_logs(self, results, peak_date):
        print("\n" + "="*80)
        print(f"INITIAL LOG: HEAT CYCLE DETECTED ON: {peak_date}")
        print(f"LOGIC: Sequential Pattern Match (Thermal Jump -> Behavioral Persistence)")
        print("="*80)

        results.sort(key=lambda x: x['date'])
        for r in results:
            if r['date'] == peak_date:
                status = "ALERT: CONFIRMED HEAT"
                if r['stress']: status = "SUPPRESSED (STRESS)"
            elif r['score'] > 25:
                status = "LOG: PROESTRUS"
            else:
                status = "NORMAL"

            print(f"[{r['date']}] {status:<22} | Night Spike: {r['spike']:.2f}C | Persistence: {round(r['res']*100)}%")

# --- PIPELINE EXECUTION ---
system = CattleIntelligenceSystem()
files = sorted(glob.glob(file_pattern))

if not files:
    print("Error: No data files found.")
else:
    raw_data = pd.concat([pd.read_csv(f) for f in files])
    raw_data['timestamp_ist'] = pd.to_datetime(raw_data['timestamp_ist'])

    # Stage 1: Feature Extraction
    features = system.extract_features(raw_data)
    # Stage 2: Activity Recognition
    ar_output = system.predict_activity(features)

    # Hourly Resampling
    hourly = ar_output.set_index('timestamp_ist').resample('h').agg({'temperature_value': 'mean'})
    hourly.columns = ['temp_mean']
    hourly['res_ratio'] = ar_output.set_index('timestamp_ist')['activity_class'].apply(lambda x: 1 if x == 'RES' else 0).resample('h').mean()

    # Stage 3: Env Check
    is_stress = system.check_heat_stress(hourly)

    # Stage 4 & 5: Health Logic and Alerts
    results, peak_date = system.cattle_logic_engine(hourly, is_stress)
    system.generate_logs(results, peak_date)


INITIAL LOG: HEAT CYCLE DETECTED ON: 2026-01-28
LOGIC: Sequential Pattern Match (Thermal Jump -> Behavioral Persistence)
[2026-01-28] ALERT: CONFIRMED HEAT  | Night Spike: 0.00C | Persistence: 2%
[2026-01-29] NORMAL                 | Night Spike: 0.00C | Persistence: 0%
[2026-01-30] NORMAL                 | Night Spike: 0.00C | Persistence: 0%
[2026-01-31] NORMAL                 | Night Spike: 0.00C | Persistence: 0%


In [ ]:
# Testing on same cow data from 18th to 28th Jan (assumption that before 21-24 days the same heat signature has been observed
data_path1 = '/content/drive/MyDrive/slot1/'
#files = sorted(glob.glob(os.path.join(data_path, 'node-184-2026-02-*.csv')))
file_pattern1 = os.path.join(data_path1, 'node-184-2026-01-*.csv')

import pandas as pd
import numpy as np
import glob
import os

class CattleIntelligenceSystem:
    def __init__(self):
        # Configuration for scalability and precision
        self.RES_THRESHOLD = 0.35
        self.NIGHT_HOURS = [23, 0, 1, 2, 3]
        self.SOLAR_HOURS = [11, 12, 13, 14, 15, 16]

    # --- BLOCK 1: FEATURE EXTRACTION ---
    def extract_features(self, df):
        df = df.copy()
        df['mag'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)
        df['vedba'] = np.abs(df['mag'] - df['mag'].rolling(window=50, center=True).mean())
        return df.fillna(0)

    # --- BLOCK 2: AR MODEL (Activity Recognition) ---
    def predict_activity(self, df):
        conditions = [(df['vedba'] > self.RES_THRESHOLD), (df['vedba'] > 0.15)]
        df['activity_class'] = np.select(conditions, ['RES', 'FEED'], default='STANDING')
        return df

    # --- BLOCK 3: ENVIRONMENTAL HEAT STRESS BLOCK ---
    def check_heat_stress(self, hourly_df):
        solar_data = hourly_df[hourly_df.index.hour.isin(self.SOLAR_HOURS)]
        # Flag environmental stress if daytime avg is extreme
        return solar_data['temp_mean'].mean() > 40.5

    # --- BLOCK 4: CATTLE LOGIC (The Core Detector) ---
    def cattle_logic_engine(self, hourly_df, env_stress):
        # GLOBAL ANCHOR: The lowest resting temperature of the whole period
        # This preserves the 9.11C delta reported for Feb 13
        global_night_base = hourly_df[hourly_df.index.hour.isin([0, 1, 2])]['temp_mean'].min()

        daily_stats = []
        for day, day_data in hourly_df.groupby(hourly_df.index.date):
            # 1. Biological Spike: Night Peak vs Global Anchor
            night_max = day_data[day_data.index.hour.isin(self.NIGHT_HOURS)]['temp_mean'].max()
            night_spike = night_max - global_night_base

            # 2. Behavioral Persistence: 3-hour sustained restlessness
            persistence = day_data['res_ratio'].rolling(3, min_periods=1).mean().max()

            # 3. Decision Score
            score = (night_spike * 15) + (persistence * 40)

            daily_stats.append({
                'date': str(day),
                'spike': max(0, night_spike),
                'res': persistence if not np.isnan(persistence) else 0,
                'score': score,
                'stress': env_stress
            })

        # Rank by score to identify the single most clinical day
        daily_stats.sort(key=lambda x: x['score'], reverse=True)
        peak_date = daily_stats[0]['date']
        return daily_stats, peak_date

    # --- BLOCK 5: ALERT & LOGGING ---
    def generate_logs(self, results, peak_date):
        print("\n" + "="*80)
        print(f"INITIAL LOG: HEAT CYCLE DETECTED ON: {peak_date}")
        print(f"LOGIC: Sequential Pattern Match (Thermal Jump -> Behavioral Persistence)")
        print("="*80)

        results.sort(key=lambda x: x['date'])
        for r in results:
            if r['date'] == peak_date:
                status = "ALERT: CONFIRMED HEAT"
                if r['stress']: status = "SUPPRESSED (STRESS)"
            elif r['score'] > 25:
                status = "LOG: PROESTRUS"
            else:
                status = "NORMAL"

            print(f"[{r['date']}] {status:<22} | Night Spike: {r['spike']:.2f}C | Persistence: {round(r['res']*100)}%")

# --- PIPELINE EXECUTION ---
system = CattleIntelligenceSystem()
files = sorted(glob.glob(file_pattern1))

if not files:
    print("Error: No data files found.")
else:
    raw_data = pd.concat([pd.read_csv(f) for f in files])
    raw_data['timestamp_ist'] = pd.to_datetime(raw_data['timestamp_ist'])

    # Stage 1: Feature Extraction
    features = system.extract_features(raw_data)
    # Stage 2: Activity Recognition
    ar_output = system.predict_activity(features)

    # Hourly Resampling
    hourly = ar_output.set_index('timestamp_ist').resample('h').agg({'temperature_value': 'mean'})
    hourly.columns = ['temp_mean']
    hourly['res_ratio'] = ar_output.set_index('timestamp_ist')['activity_class'].apply(lambda x: 1 if x == 'RES' else 0).resample('h').mean()

    # Stage 3: Env Check
    is_stress = system.check_heat_stress(hourly)

    # Stage 4 & 5: Health Logic and Alerts
    results, peak_date = system.cattle_logic_engine(hourly, is_stress)
    system.generate_logs(results, peak_date)




INITIAL LOG: HEAT CYCLE DETECTED ON: 2026-01-21
LOGIC: Sequential Pattern Match (Thermal Jump -> Behavioral Persistence)
[2026-01-18] NORMAL                 | Night Spike: 0.06C | Persistence: 35%
[2026-01-19] NORMAL                 | Night Spike: 0.06C | Persistence: 50%
[2026-01-20] NORMAL                 | Night Spike: 0.08C | Persistence: 32%
[2026-01-21] ALERT: CONFIRMED HEAT  | Night Spike: 0.07C | Persistence: 83%
[2026-01-22] NORMAL                 | Night Spike: 0.03C | Persistence: 34%
[2026-01-23] NORMAL                 | Night Spike: 0.05C | Persistence: 28%
[2026-01-24] NORMAL                 | Night Spike: 0.03C | Persistence: 60%
[2026-01-25] NORMAL                 | Night Spike: 0.07C | Persistence: 35%
[2026-01-26] NORMAL                 | Night Spike: 0.06C | Persistence: 38%
[2026-01-27] LOG: PROESTRUS         | Night Spike: 0.04C | Persistence: 65%
[2026-01-28] NORMAL                 | Night Spike: 0.00C | Persistence: 12%
